In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import json, os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

BASE_DIR = "."
WORK_DIR = "."
# ── Data laden ────────────────────────────────────────────────────────────────
from collections import Counter

dataset = json.load(open(os.path.join(BASE_DIR, "train_dataset.json"), "r"))

def get_bucket(pct):
    if pct >= 80: return "PERFECTE"
    if pct >= 60: return "GOEDE"
    if pct >= 40: return "GEMIDDELDE"
    return "LAGE"

buckets = Counter(get_bucket(item["output"]["match_percentage"]) for item in dataset)
print(f"Train dataset: {len(dataset)} entries")
print(f"  LAGE      : {buckets['LAGE']}")
print(f"  GEMIDDELDE: {buckets['GEMIDDELDE']}")
print(f"  GOEDE     : {buckets['GOEDE']}")
print(f"  PERFECTE  : {buckets['PERFECTE']}")

# OpenJob — Fine-tuning Notebook
**Model:** `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`  
**Taak:** CV ↔ Vacature matching + VDAB-motivatiebrief (Dutch)  
**Dataset:** `train_dataset.json` (60 entries) | **Eval:** `test_dataset.json` (10 entries)

---
## 0. Data laden

## 1. Installatie & GPU check

In [ ]:
!pip install --upgrade unsloth unsloth_zoo
!pip install trl peft accelerate bitsandbytes

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 2. Model laden

In [ ]:
import shutil, os
if os.path.exists("/kaggle/working/unsloth_compiled_cache"):
    shutil.rmtree("/kaggle/working/unsloth_compiled_cache")
    print("Unsloth cache cleared")

from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

max_seq_length = 4096
dtype = None

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

## 3. Dataset voorbereiden

In [ ]:
from datasets import Dataset

# PROMPT_TEMPLATE — inline zodat het werkt op Kaggle zonder system_prompt.py
PROMPT_TEMPLATE = """\
Je bent een uiterst kritische AI Recruiter. Je doel is een eerlijke vergelijking te maken tussen het CV en de Vacature en een motivatiebrief te schrijven volgens de VDAB-normen.
---
{vacature}
---
{cv}
---
STRIKTE ANALYSE REGELS:
1. MATCHED SKILLS: Alleen vaardigheden die LETTERLIJK of als DIRECT SYNONIEM in beide teksten voorkomen.
2. MISSING SKILLS: Alleen concrete vereisten uit de vacature die ontbreken op het CV.
3. MATCH PERCENTAGE:
- 80-100%: Perfecte match.
- 60-79%: Goede basis, mist details.
- 40-59%: Verschillend vakgebied, maar overdraagbare technische skills.
- <40%: Geen relevante match.
RICHTLIJNEN VOOR DE VDAB-MOTIVATIEBRIEF:
Schrijf een volledige brief in het veld 'motivation_letter' met deze opbouw:
1. ONDERWERP: Maak duidelijk voor welke vacature de kandidaat solliciteert. Gebruik de exacte functietitel uit de vacature.
2. AANSPREKING: Vermeldt de vacature een contactpersoon? Gebruik dan "Geachte mevrouw [familienaam]," of "Geachte meneer [familienaam],". Geen contactpersoon? Gebruik "Beste,".
3. INLEIDING: Maak de werkgever nieuwsgierig met een sterke en persoonlijke openingszin. Verwijs naar iets specifieks aan het bedrijf of de vacature — geen generieke zin.
4. MOTIVATIE & TROEVEN:
- Leg uit waarom de kandidaat specifiek voor dít bedrijf en deze job kiest. Hoe specifieker en persoonlijker, hoe overtuigender.
- Beschrijf waarom de kandidaat de geschikte persoon is. Koppel concrete ervaringen of prestaties van het CV aan de vereisten uit de vacature.
5. AFSLUITING: Sluit af met een krachtige zin die de werkgever het laatste zetje geeft om de kandidaat uit te nodigen. Vermeld dat de kandidaat graag meer uitleg geeft tijdens een persoonlijk gesprek.
6. GROET: Eindig met "Met vriendelijke groet," gevolgd door de volledige naam van de kandidaat.
STRIKTE VOORWAARDEN:
- Gebruik NOOIT placeholders (zoals [Naam]). Als informatie ontbreekt, formuleer de zin dan zodanig dat het wegvallen niet opvalt.
- Gebruik \\n voor alle witregels en alinea-overgangen in de JSON string.
- Schrijf in professioneel, foutloos Nederlands (u-vorm)
GEEF UITSLUITEND JSON TERUG:
{{
"match_percentage": <int>,
"matched_skills": ["skill1", "skill2"],
"missing_skills": ["skill1", "skill2"],
"motivation_letter": "...",
"match_text": "[VOLLEDIGE NAAM VAN CV] heeft een [PERFECTE | GOEDE | GEMIDDELDE | LAGE] match met de vacature."
}}"""

def format_prompt(item):
    user_content = PROMPT_TEMPLATE.format(
        vacature=item["input"]["vacature"],
        cv=item["input"]["cv"],
    )
    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": json.dumps(item["output"], ensure_ascii=False)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted_data = [format_prompt(item) for item in dataset]
hf_dataset = Dataset.from_dict({"text": formatted_data})

# Verifieer token lengte (mag niet > max_seq_length zijn)
lengths = [len(tokenizer.encode(t)) for t in formatted_data]
print(f"Token lengte — min: {min(lengths)}, max: {max(lengths)}, gemiddeld: {sum(lengths)//len(lengths)}")
print(f"Entries > 4096 tokens: {sum(1 for l in lengths if l > 4096)}")
print(hf_dataset)


## 4. LoRA adapters

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

## 5. Trainer configureren

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,   # effectieve batch size = 4
        warmup_steps=5,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=1,
        dataloader_pin_memory=False,
        report_to="none",               # geen wandb/tensorboard
        average_tokens_across_devices=False,
    ),
)

## 6. Trainen

In [ ]:
# Train the model
trainer_stats = trainer.train()

## 7. Exporteren & uploaden naar HuggingFace

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
folder_name = f"open-jobs_gguf_model_{timestamp}"
print(f"Model versie: {folder_name}")

In [ ]:
import os
from huggingface_hub import login

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    hugging_face = UserSecretsClient().get_secret("open_job")
else:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname("./"), ".env"))
    hugging_face = os.getenv("open_job")

login(hugging_face)
print(f"Ingelogd als: hf_{'*' * 8}{hugging_face[-4:]}")

In [ ]:
from huggingface_hub import login

login(hugging_face)

# In Unsloth gebruik je daarna dit om te uploaden voor Ollama:
model.push_to_hub_gguf(
    "Vibe-Bataklik/open_jobs",
    tokenizer,
    quantization_method = "q4_k_m", # Aanbevolen voor Ollama
    token = hugging_face,
)

## 9. Performance evaluatie op test_dataset

In [ ]:
import json, re
from datetime import datetime

FastLanguageModel.for_inference(model)

def extract_json_safe(text):
    m = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    start = text.rfind('{')
    end   = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        return json.loads(text[start:end+1])
    return None

eval_dataset = json.load(open(os.path.join(BASE_DIR, "test_dataset.json"), "r"))
valid_json_count = 0
pct_errors      = []
bucket_correct  = 0
details         = []

print(f"Evaluating {len(eval_dataset)} samples...\n")

for i, item in enumerate(eval_dataset):
    expected_pct    = item["output"]["match_percentage"]
    expected_bucket = get_bucket(expected_pct)

    user_content = PROMPT_TEMPLATE.format(
        vacature=item["input"]["vacature"],
        cv=item["input"]["cv"],
    )

    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([chat_text], return_tensors="pt").to("cuda")
    input_len = inputs.input_ids.shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=2048,
            use_cache=True,  # Snel — vereist kernel restart na pip upgrade
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response_text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    parsed = extract_json_safe(response_text)

    row = {
        "sample_index"    : i,
        "expected_pct"    : expected_pct,
        "expected_bucket" : expected_bucket,
        "valid_json"      : False,
        "predicted_pct"   : None,
        "predicted_bucket": None,
        "pct_error"       : None,
        "bucket_correct"  : False,
    }

    if parsed and "match_percentage" in parsed:
        valid_json_count += 1
        pred_pct    = parsed["match_percentage"]
        pred_bucket = get_bucket(pred_pct)
        error       = abs(pred_pct - expected_pct)
        row.update({
            "valid_json"      : True,
            "predicted_pct"   : pred_pct,
            "predicted_bucket": pred_bucket,
            "pct_error"       : error,
            "bucket_correct"  : pred_bucket == expected_bucket,
        })
        pct_errors.append(error)
        if pred_bucket == expected_bucket:
            bucket_correct += 1

    details.append(row)
    status = f"{row['predicted_pct']}% ({row['predicted_bucket']})" if row["valid_json"] else "INVALID JSON"
    mark   = "✓" if row["bucket_correct"] else "✗"
    print(f"  {mark} [{i+1:2d}] expected={expected_pct:3d}% ({expected_bucket:10s}) | predicted={status}")

n = len(eval_dataset)
metrics = {
    "model_name"              : folder_name,
    "evaluated_at"            : datetime.now().isoformat(),
    "num_samples"             : n,
    "valid_json_rate"         : round(valid_json_count / n, 3),
    "mean_absolute_error_pct" : round(sum(pct_errors) / len(pct_errors), 1) if pct_errors else None,
    "bucket_accuracy"         : round(bucket_correct / n, 3),
    "details"                 : details,
}

perf_file = os.path.join(WORK_DIR, f"performance_{folder_name}.json")
with open(perf_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\n{'='*50}")
print(f"  Model        : {folder_name}")
print(f"  Valid JSON   : {metrics['valid_json_rate']*100:.0f}%  ({valid_json_count}/{n})")
print(f"  Bucket acc.  : {metrics['bucket_accuracy']*100:.0f}%  ({bucket_correct}/{n})")
print(f"  Mean abs err : {metrics['mean_absolute_error_pct']}% match-verschil")
print(f"  Opgeslagen   : {perf_file}")